In [1]:

import aisuite as ai
import pandas as pd
import json
import requests
from bs4 import BeautifulSoup

url = "https://jobpaw.com/professionals/job-details?idj=17477"

job_urls = []

for job_id in range(16500,17498):
    url=f"https://jobpaw.com/professionals/job-details?idj={job_id}"
    job_urls.append(url)

job_urls
    

['https://jobpaw.com/professionals/job-details?idj=16500',
 'https://jobpaw.com/professionals/job-details?idj=16501',
 'https://jobpaw.com/professionals/job-details?idj=16502',
 'https://jobpaw.com/professionals/job-details?idj=16503',
 'https://jobpaw.com/professionals/job-details?idj=16504',
 'https://jobpaw.com/professionals/job-details?idj=16505',
 'https://jobpaw.com/professionals/job-details?idj=16506',
 'https://jobpaw.com/professionals/job-details?idj=16507',
 'https://jobpaw.com/professionals/job-details?idj=16508',
 'https://jobpaw.com/professionals/job-details?idj=16509',
 'https://jobpaw.com/professionals/job-details?idj=16510',
 'https://jobpaw.com/professionals/job-details?idj=16511',
 'https://jobpaw.com/professionals/job-details?idj=16512',
 'https://jobpaw.com/professionals/job-details?idj=16513',
 'https://jobpaw.com/professionals/job-details?idj=16514',
 'https://jobpaw.com/professionals/job-details?idj=16515',
 'https://jobpaw.com/professionals/job-details?idj=16516

In [2]:
def extract_text_from_url(url):
    html = requests.get(url).text
    soup = BeautifulSoup(html, "html.parser")

    # enlever le bruit
    for tag in soup(["script", "style", "nav", "footer", "header"]):
        tag.decompose()

    text = soup.get_text(separator="\n", strip=True)
    return text


In [3]:
def build_prompt(text):
    return f"""
Tu es un assistant chargé d'extraire des informations structurées
à partir d'une offre d'emploi.

Texte de l'offre :
---
{text}
---

Retourne UNIQUEMENT un JSON valide avec le format exact suivant :
{{
  "titre_du_poste": "",
  "compagnie": "",
  "date_publication": "",
  "ville": "",
  "domaine": "",
  "specialité":"",
  "diplome_requis": "",
  "annee_experience": "",
  "type_contrat": "",
  "competences":""
}}

Règles :
- Si une information est absente ou ambiguë, mets null
- diplome_requis ∈ ["Licence", "Master", "Doctorat", "Non précisé"]
-type_contrat ∈ ["CDD","CDI"]
- annee_experience :
    - entier
    - si intervalle, prendre la valeur minimale
    - null si non précisé
- competences :
    - liste les compétences demandées, si il y a plusieurs, liste les 5 premieres. 
- Ne mets RIEN en dehors du JSON
"""



In [5]:
client = ai.Client()

def extract_job_json(text):
    prompt = build_prompt(text)

    response = client.chat.completions.create(
        model="openai:gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}]
    )

    json_output = response.choices[0].message.content
    return json.loads(json_output)


In [6]:

results = []

for url in job_urls:
    print(f"Traitement de : {url}")

    try:
        text = extract_text_from_url(url)
        job_data = extract_job_json(text)
        job_data["source_url"] = url  # utile pour debug
        results.append(job_data)

    except Exception as e:
        print(f"Erreur pour {url} : {e}")


Traitement de : https://jobpaw.com/professionals/job-details?idj=16500
Traitement de : https://jobpaw.com/professionals/job-details?idj=16501
Traitement de : https://jobpaw.com/professionals/job-details?idj=16502
Traitement de : https://jobpaw.com/professionals/job-details?idj=16503
Traitement de : https://jobpaw.com/professionals/job-details?idj=16504
Traitement de : https://jobpaw.com/professionals/job-details?idj=16505
Traitement de : https://jobpaw.com/professionals/job-details?idj=16506
Traitement de : https://jobpaw.com/professionals/job-details?idj=16507
Traitement de : https://jobpaw.com/professionals/job-details?idj=16508
Traitement de : https://jobpaw.com/professionals/job-details?idj=16509
Traitement de : https://jobpaw.com/professionals/job-details?idj=16510
Traitement de : https://jobpaw.com/professionals/job-details?idj=16511
Traitement de : https://jobpaw.com/professionals/job-details?idj=16512
Traitement de : https://jobpaw.com/professionals/job-details?idj=16513
Traite

In [8]:

df = pd.DataFrame(results)

ValueError: time data "7 Aout 2024" doesn't match format "%d %b %Y", at position 0. You might want to try:
    - passing `format` if your strings have a consistent format;
    - passing `format='ISO8601'` if your strings are all ISO8601 but not necessarily in exactly the same format;
    - passing `format='mixed'`, and the format will be inferred for each element individually. You might want to use `dayfirst` alongside this.

In [10]:
df = pd.DataFrame(results)

In [11]:
df.to_csv("offres_emploi_haiti.csv", index=False, encoding="utf-8")


In [12]:
df.head()

,titre_du_poste,compagnie,date_publication,ville,domaine,specialité,diplome_requis,annee_experience,type_contrat,competences,source_url
0,Officier.ere ACAT et Promotion à l’Hygiène,HELVETAS Swiss Intercooperation/Haiti,7 Aout 2024,Jacmel,Sciences Humaines et Sociales,Sciences Sociales,Non précisé,4.0,CDD,[Diplôme universitaire ou certificat en scienc...,https://jobpaw.com/professionals/job-details?i...
1,Facilitateur.trice,HELVETAS Swiss Intercooperation/Haiti,7 Aout 2024,Jacmel,Sciences Humaines et Sociales,Sciences Sociales,Licence,3.0,CDD,[Diplôme universitaire ou certificat en scienc...,https://jobpaw.com/professionals/job-details?i...
2,Officier Construction EPA,HELVETAS Swiss Intercooperation/Haiti,7 Aout 2024,Jacmel,Sciences de l’Ingénieur,Génie Civil,Non précisé,3.0,CDD,[Diplôme universitaire en génie civil ou dans ...,https://jobpaw.com/professionals/job-details?i...
3,Officier/re des Ressources Humaines,Institut de Formation du Sud,8 Aout 2024,Delmas,"Management/Gestion, Finance, Comptabilité et C...",Gestion des Ressources Humaines,Licence,NaN,CDD,"[Bonne connaissance du Français et du Créole, ...",https://jobpaw.com/professionals/job-details?i...
4,Senior Technical Specialist HIV/TB,Interchurch Medical Assistance World Health,8 Aout 2024,Port-au-Prince,Santé et Professions médicales,Management et Gestion des services de santé,Master,5.0,CDI,"[Intégrité, Engagement, Rigueur et sens des re...",https://jobpaw.com/professionals/job-details?i...
